<a href="https://colab.research.google.com/github/Gianluca-dot/Machine_Learning_Operations_Sentiment_Monitoring/blob/main/Progetto_di_Machine_Learning_Operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Setup delle Cartelle e Dipendenze

In [1]:
import os
import subprocess

# 1. Creazione della struttura delle cartelle
os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs(".github/workflows", exist_ok=True)

# 2. Scrittura del file requirements.txt con streamlit inclusa
requirements_content = """transformers>=4.30.0
torch>=2.0.0
datasets>=2.12.0
scikit-learn>=1.2.0
pandas>=2.0.0
pytest>=7.3.0
streamlit>=1.20.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("File requirements.txt creato con successo!")

# 3. Installazione delle dipendenze
subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)

File requirements.txt creato con successo!


CompletedProcess(args=['pip', 'install', '-r', 'requirements.txt'], returncode=0)

##Classe SentimentAnalyzer (cardiffnlp)

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

class SentimentAnalyzer:
    def __init__(self, model_name: str = MODEL_NAME):
        """Inizializza e carica il tokenizer e il modello pre-addestrato RoBERTa."""
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.labels = ["negative", "neutral", "positive"]

    def preprocess_text(self, text: str) -> str:
        """Normalizza il testo dei social media sostituendo username ed URL."""
        text = re.sub(r"http\S+|www\S+|https\S+", "http", text, flags=re.MULTILINE)
        text = re.sub(r"@\w+", "@user", text)
        return text.strip()

    def predict(self, text: str) -> dict:
        """Effettua l'inferenza restituendo il sentiment predetto e la confidenza."""
        cleaned_text = self.preprocess_text(text)
        inputs = self.tokenizer(cleaned_text, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)
            scores = outputs.logits[0]
            probabilities = torch.softmax(scores, dim=0)

        confidence, predicted_class_id = torch.max(probabilities, dim=0)
        predicted_label = self.labels[predicted_class_id.item()]

        return {
            "text": text,
            "cleaned_text": cleaned_text,
            "sentiment": predicted_label,
            "confidence": round(confidence.item(), 4),
            "probabilities": {
                label: round(prob.item(), 4) for label, prob in zip(self.labels, probabilities)
            }
        }

if __name__ == "__main__":
    analyzer = SentimentAnalyzer()
    sample_text = "MachineInnovators Inc. releases incredible new AI tools! Great job @team https://example.com"
    result = analyzer.predict(sample_text)
    print("Risultato dell'inferenza di prova:")
    print(result)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Risultato dell'inferenza di prova:
{'text': 'MachineInnovators Inc. releases incredible new AI tools! Great job @team https://example.com', 'cleaned_text': 'MachineInnovators Inc. releases incredible new AI tools! Great job @user http', 'sentiment': 'positive', 'confidence': 0.9879, 'probabilities': {'negative': 0.0025, 'neutral': 0.0096, 'positive': 0.9879}}


##Download del Dataset e Creazione Campione

In [3]:
import os
import pandas as pd
from datasets import load_dataset

print("Download del dataset 'cardiffnlp/tweet_eval' (sentiment) da HuggingFace...")
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Mappatura delle etichette del dataset: 0: negative, 1: neutral, 2: positive
label_map = {0: "negative", 1: "neutral", 2: "positive"}

# Estrazione del set di test
test_df = pd.DataFrame(dataset["test"])
test_df["sentiment"] = test_df["label"].map(label_map)

# Salvataggio di un campione da 200 righe
output_path = os.path.join("data", "test_sample.csv")
test_df[["text", "sentiment"]].head(200).to_csv(output_path, index=False)
print(f"Campione di dati salvato con successo in '{output_path}'.")

Download del dataset 'cardiffnlp/tweet_eval' (sentiment) da HuggingFace...


README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

...

sentiment/validation-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Campione di dati salvato con successo in 'data/test_sample.csv'.


##Valutazione Offline e Salvataggio metrics.json


In [4]:
import json
import os
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def evaluate_model(data_path: str = "data/test_sample.csv"):
    df = pd.read_csv(data_path)
    analyzer = SentimentAnalyzer()

    y_true = df["sentiment"].tolist()
    y_pred = []

    print(f"Valutazione in corso su {len(df)} esempi...")
    for text in df["text"]:
        prediction = analyzer.predict(text)
        y_pred.append(prediction["sentiment"])

    # 1. Calcolo delle metriche
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    cm = confusion_matrix(y_true, y_pred, labels=["negative", "neutral", "positive"])

    # 2. Stampa a schermo
    print("\n--- RISULTATI VALUTAZIONE ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print("\nMatrice di Confusione (Righe: Reali, Colonne: Predetti):")
    print("Etichette: ['negative', 'neutral', 'positive']")
    print(cm)
    print("\nReport di Classificazione Dettagliato:")
    print(classification_report(y_true, y_pred))

    # 3. Salvataggio delle metriche su file JSON
    metrics = {
        "accuracy": round(acc, 4),
        "f1_macro": round(f1_macro, 4),
        "f1_weighted": round(f1_weighted, 4)
    }

    metrics_path = "data/metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)

    print(f"\nMetriche salvate con successo in '{metrics_path}'.")

evaluate_model()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Valutazione in corso su 200 esempi...

--- RISULTATI VALUTAZIONE ---
Accuracy: 0.7350
F1-Score (Macro): 0.7420
F1-Score (Weighted): 0.7351

Matrice di Confusione (Righe: Reali, Colonne: Predetti):
Etichette: ['negative', 'neutral', 'positive']
[[48 10  1]
 [24 64  9]
 [ 3  6 35]]

Report di Classificazione Dettagliato:
              precision    recall  f1-score   support

    negative       0.64      0.81      0.72        59
     neutral       0.80      0.66      0.72        97
    positive       0.78      0.80      0.79        44

    accuracy                           0.73       200
   macro avg       0.74      0.76      0.74       200
weighted avg       0.75      0.73      0.74       200


Metriche salvate con successo in 'data/metrics.json'.


##Script di Fine-Tuning

In [5]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

def train():
    print("Inizializzazione dello script di Retraining/Fine-Tuning...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

    dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

    print("Tokenizzazione del dataset...")
    tokenized_datasets = dataset.map(tokenize_function, batched=True)

    use_cuda = torch.cuda.is_available()

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        load_best_model_at_end=True,
        use_cpu=not use_cuda,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"].select(range(500)),
        eval_dataset=tokenized_datasets["validation"].select(range(100)),
    )

    print("Avvio del retraining...")
    trainer.train()

    output_model_dir = "./retrained_model"
    model.save_pretrained(output_model_dir)
    tokenizer.save_pretrained(output_model_dir)
    print(f"Modello ri-addestrato salvato con successo in '{output_model_dir}'.")

if __name__ == "__main__":
    train()

Inizializzazione dello script di Retraining/Fine-Tuning...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizzazione del dataset...


Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

...

Avvio del retraining...


Epoch,Training Loss,Validation Loss
1,0.724040,0.591310


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modello ri-addestrato salvato con successo in './retrained_model'.


##Scrittura ed Esecuzione Test Unitari (pytest)

In [6]:
import subprocess

# 1. Scrittura di tests/test_predict.py
test_predict_code = """import pytest
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

class SentimentAnalyzer:
    def __init__(self, model_name: str = MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.labels = ["negative", "neutral", "positive"]

    def preprocess_text(self, text: str) -> str:
        text = re.sub(r"http\S+|www\S+|https\S+", "http", text, flags=re.MULTILINE)
        text = re.sub(r"@\w+", "@user", text)
        return text.strip()

    def predict(self, text: str) -> dict:
        cleaned_text = self.preprocess_text(text)
        inputs = self.tokenizer(cleaned_text, return_tensors="pt", truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)
            scores = outputs.logits[0]
            probabilities = torch.softmax(scores, dim=0)

        confidence, predicted_class_id = torch.max(probabilities, dim=0)
        predicted_label = self.labels[predicted_class_id.item()]

        return {
            "text": text,
            "cleaned_text": cleaned_text,
            "sentiment": predicted_label,
            "confidence": round(confidence.item(), 4),
            "probabilities": {
                label: round(prob.item(), 4) for label, prob in zip(self.labels, probabilities)
            }
        }

@pytest.fixture(scope="module")
def analyzer():
    return SentimentAnalyzer()

def test_preprocessing(analyzer):
    raw_text = "Check this link https://example.com and mention @john!"
    cleaned = analyzer.preprocess_text(raw_text)
    assert "https://example.com" not in cleaned
    assert "@john" not in cleaned
    assert "http" in cleaned
    assert "@user" in cleaned

def test_predict_output_structure(analyzer):
    result = analyzer.predict("This product is amazing!")
    assert isinstance(result, dict)
    assert "sentiment" in result
    assert "confidence" in result

def test_sentiment_values(analyzer):
    result = analyzer.predict("Great service!")
    assert result["sentiment"] in ["negative", "neutral", "positive"]
    assert 0.0 <= result["confidence"] <= 1.0
"""

with open("tests/test_predict.py", "w") as f:
    f.write(test_predict_code)

# 2. Scrittura di tests/test_evaluate.py
test_evaluate_code = """import os
import json

def test_metrics_file_creation():
    metrics_path = "data/metrics.json"
    assert os.path.exists(metrics_path), "Il file data/metrics.json non esiste."

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    assert "accuracy" in metrics
    assert "f1_macro" in metrics
    assert "f1_weighted" in metrics
"""

with open("tests/test_evaluate.py", "w") as f:
    f.write(test_evaluate_code)

print("File di test creati con successo.")

# 3. Esecuzione di pytest
subprocess.run(["pytest", "tests/"], check=True)

<>:18: SyntaxWarning: invalid escape sequence '\S'
<>:18: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_6575/3549091308.py:18: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub(r"http\S+|www\S+|https\S+", "http", text, flags=re.MULTILINE)


File di test creati con successo.


CompletedProcess(args=['pytest', 'tests/'], returncode=0)

##Generazione Workflow CI/CD (.github/workflows/ci_cd.yml)

In [7]:
cd_workflow_yaml_content = """name: Sentiment Analysis MLOps CI/CD Pipeline

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout del codice
      uses: actions/checkout@v3

    - name: Configurazione Python
      uses: actions/setup-python@v4
      with:
        python-version: "3.10"

    - name: Cache delle dipendenze pip
      uses: actions/cache@v3
      with:
        path: ~/.cache/pip
        key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
        restore-keys: |
          ${{ runner.os }}-pip-

    - name: Installazione Dipendenze
      run: |
        python -m pip install --upgrade pip
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Esecuzione Unit Test con Pytest
      run: |
        pytest tests/
"""

with open(".github/workflows/ci_cd.yml", "w") as f:
    f.write(cd_workflow_yaml_content)

print("File '.github/workflows/ci_cd.yml' generato correttamente.")

File '.github/workflows/ci_cd.yml' generato correttamente.


##Applicazione Streamlit (app_streamlit.py)

In [8]:
streamlit_code = """import streamlit as st
import pandas as pd
import time
import os
from datetime import datetime
from transformers import pipeline

st.set_page_config(
    page_title="MLOps - Sentiment Analysis",
    page_icon="📊",
    layout="wide"
)

LOG_FILE = "predictions.csv"

if not os.path.exists(LOG_FILE):
    df_init = pd.DataFrame(columns=["timestamp", "text", "prediction", "confidence", "latency_ms"])
    df_init.to_csv(LOG_FILE, index=False)

@st.cache_resource
def load_sentiment_model():
    return pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

classifier = load_sentiment_model()

st.title("📊 MLOps Sentiment Analysis & Live Monitoring")

tab_deploy, tab_monitoring = st.tabs(["🚀 Deploy & Inferenza", "📈 Dashboard Monitoraggio"])

with tab_deploy:
    st.header("Interfaccia di Predizione Sentiment")
    user_input = st.text_area("Testo di input (Inglese):", value="", height=120)

    if st.button("Analizza Sentiment", type="primary"):
        if not user_input.strip():
            st.warning("Inserisci un testo valido.")
        else:
            start_time = time.time()
            result = classifier(user_input)[0]
            latency = round((time.time() - start_time) * 1000, 2)

            label = result["label"].upper()
            confidence = round(float(result["score"]), 4)
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            col1, col2, col3 = st.columns(3)
            with col1:
                st.success(f"Esito Predizione: {label}")
            with col2:
                st.metric("Confidenza Modello", f"{confidence * 100:.2f}%")
            with col3:
                st.metric("Latenza Sistema", f"{latency} ms")

            new_log = pd.DataFrame([{
                "timestamp": timestamp,
                "text": user_input,
                "prediction": label,
                "confidence": confidence,
                "latency_ms": latency
            }])
            new_log.to_csv(LOG_FILE, mode="a", header=False, index=False)
            st.toast("Predizione registrata nei log di monitoraggio.")

with tab_monitoring:
    st.header("Dashboard di Monitoraggio Continuo")

    if os.path.exists(LOG_FILE):
        df_logs = pd.read_csv(LOG_FILE)

        if df_logs.empty:
            st.info("Nessun dato presente nel registro log.")
        else:
            total_requests = len(df_logs)
            avg_confidence = round(df_logs["confidence"].mean() * 100, 2)
            avg_latency = round(df_logs["latency_ms"].mean(), 2)

            m1, m2, m3 = st.columns(3)
            m1.metric("Totale Richieste", total_requests)
            m2.metric("Confidenza Media", f"{avg_confidence}%")
            m3.metric("Latenza Media", f"{avg_latency} ms")

            st.markdown("---")
            col_g1, col_g2 = st.columns(2)

            with col_g1:
                st.subheader("Distribuzione Output (Concept Drift)")
                st.bar_chart(df_logs["prediction"].value_counts())

            with col_g2:
                st.subheader("Andamento Latenza (ms)")
                st.line_chart(df_logs["latency_ms"])

            st.subheader("📋 Registro Log di Inferenza")
            st.dataframe(df_logs.sort_values(by="timestamp", ascending=False), use_container_width=True)
"""

with open("app_streamlit.py", "w") as f:
    f.write(streamlit_code)

print("File app_streamlit.py creato con successo!")

File app_streamlit.py creato con successo!


##Generazione Log di Prova per predictions.csv

In [9]:
import pandas as pd
import time
from datetime import datetime

# Generazione di 10 log di prova con il modello ufficiale
analyzer = SentimentAnalyzer()

sample_sentences = [
    "I absolute love this new MLOps pipeline! Works great.",
    "The system works as expected, nothing special.",
    "This is worst performance I have ever seen.",
    "Pretty good results overall.",
    "The application crashed twice today.",
    "I am neutral about this update.",
    "Awesome feature! High accuracy.",
    "Extremely slow response time.",
    "Just testing the logging system.",
    "Fantastic model deployment!"
]

logs = []
for text in sample_sentences:
    start_time = time.time()
    res = analyzer.predict(text)
    latency = round((time.time() - start_time) * 1000, 2)

    logs.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "text": text,
        "prediction": res["sentiment"].upper(),
        "confidence": res["confidence"],
        "latency_ms": latency
    })

df_test_logs = pd.DataFrame(logs)
df_test_logs.to_csv("predictions.csv", index=False)
print("✅ Generati 10 log di test in 'predictions.csv'")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Generati 10 log di test in 'predictions.csv'


##Relazione sulla Valutazione del Modello

### Relazione sulla Valutazione e il Monitoraggio del Modello di Sentiment Analysis (MLOps)

Il progetto MLOps per l'analisi del sentiment è stato strutturato per garantire una valutazione robusta del modello sia in fase di sviluppo (offline) che durante l'operatività (online).

## 1. Valutazione Offline del Modello
Valutazione intrinseca del modello di sentiment analysis (`cardiffnlp/twitter-roberta-base-sentiment-latest`) sul dataset di test (`data/test_sample.csv`).

* **Accuracy**: Percentuale di previsioni corrette totali.
* **F1-Score (Macro e Weighted)**: Misura dell'accuratezza per gestire classi sbilanciate.
* **Matrice di Confusione**: Dettaglio delle previsioni corrette e degli errori per classe (`negative`, `neutral`, `positive`).
* **Memorizzazione**: Le metriche vengono salvate in `data/metrics.json` per garantire la tracciabilità tra le versioni.

## 2. Monitoraggio Online in Tempo Reale
Monitoraggio operativo tramite l'applicazione Streamlit (`app_streamlit.py`).

* **Totale Richieste**: Conteggio delle inferenze eseguite.
* **Confidenza Media**: Fiducia del modello nelle sue predizioni.
* **Latenza Media**: Tempo impiegato per generare ciascuna predizione.
* **Distribuzione Output (Concept Drift)**: Visualizzazione delle frequenze per intercettare variazioni nella distribuzione dei dati.
* **Registro Log**: Salvataggio continuo nel file `predictions.csv`.

##Sincronizzazione Finale con GitHub (getpass)

In [11]:
import os
import subprocess
from getpass import getpass

# 1. Credenziali
REPO_URL = "github.com/Gianluca-dot/Machine_Learning_Operations_Sentiment_Monitoring.git"
USER_NAME = "Gianluca-dot"

# 2. Token
GH_TOKEN = None
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    pass

if not GH_TOKEN:
    GH_TOKEN = getpass("Incolla qui il TOKEN e premi INVIO: ").strip()

# 3. Ignora file pesanti
with open(".gitignore", "w") as f:
    f.write("retrained_model/\nresults/\n*.safetensors\n*.bin\n*.pt\n__pycache__/\n.pytest_cache/\n")

print("1. Gitignore ok")

# 4. Git Init e Add
if not os.path.exists(".git"):
    subprocess.run(["git", "init"], check=True)
    subprocess.run(["git", "branch", "-M", "main"], check=True)

subprocess.run(["git", "config", "--global", "user.name", USER_NAME], check=True)
subprocess.run(["git", "config", "--global", "user.email", "gianluca@example.com"], check=True)

subprocess.run(["git", "add", "."], check=True)
print("2. File aggiunti al push")

subprocess.run(["git", "commit", "-m", "FEAT: Sync finale MLOps"], check=True)
print("3. Commit eseguito")

# 5. Remote e Push
authenticated_url = f"https://{GH_TOKEN}@{REPO_URL}"
subprocess.run(["git", "remote", "remove", "origin"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "remote", "add", "origin", authenticated_url], check=True)

print("4. Inizio Push su GitHub...")
push_result = subprocess.run(["git", "push", "-u", "origin", "main", "--force"], capture_output=True, text=True)

if push_result.returncode == 0:
    print("✅ Push completato con successo!")
else:
    print("❌ Errore Push:", push_result.stderr)

1. Gitignore ok
2. File aggiunti al push
3. Commit eseguito
4. Inizio Push su GitHub...
✅ Push completato con successo!
